# 01 - Preparacion del Dataset

Descarga el dataset desde HuggingFace, valida imagenes, deduplica por MD5 y crea splits train/val.

Dataset: [`alejandroramirezucb/soybean_image_dataset`](https://huggingface.co/datasets/alejandroramirezucb/soybean_image_dataset)


In [ ]:
!pip install -q huggingface_hub Pillow tqdm scikit-learn

In [ ]:
import os
import hashlib
import shutil
import random
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from huggingface_hub import snapshot_download

random.seed(42)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

snapshot_download(
    repo_id="alejandroramirezucb/soybean_image_dataset",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
)
print("Descarga completada:", DATA_DIR)


In [ ]:
def validar(p, min_size=100):
    try:
        img = Image.open(p)
        img.verify()
        img = Image.open(p)
        if img.mode != "RGB":
            return False
        w, h = img.size
        return w >= min_size and h >= min_size
    except Exception:
        return False


def md5(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def contar(base):
    base = Path(base)
    if not base.exists():
        print(f"  (no existe: {base})")
        return
    for clase in sorted(base.iterdir()):
        if clase.is_dir():
            n = sum(1 for _ in clase.rglob("*") if _.is_file())
            print(f"  {clase.name:25s}: {n:>5}")


print("Train binaria:")
contar(DATA_DIR / "Train" / "clasificacion_binaria")
print("Train patogeno:")
contar(DATA_DIR / "Train" / "clasificacion_patogeno")
print("Test binaria:")
contar(DATA_DIR / "Test" / "clasificacion_binaria")
print("Test patogeno:")
contar(DATA_DIR / "Test" / "clasificacion_patogeno")


In [ ]:
CLEAN = Path("./clean")
CLEAN.mkdir(exist_ok=True)


def deduplicar_y_limpiar(origen: Path, destino: Path):
    destino.mkdir(parents=True, exist_ok=True)
    hashes = set()
    archivos = [p for p in origen.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    random.shuffle(archivos)
    keep = 0
    for p in tqdm(archivos, desc=str(origen.name)):
        if not validar(p):
            continue
        h = md5(p)
        if h in hashes:
            continue
        hashes.add(h)
        shutil.copy2(p, destino / f"{h[:8]}_{p.name}")
        keep += 1
    return keep


for split in ["Train", "Test"]:
    for tarea in ["clasificacion_binaria", "clasificacion_patogeno"]:
        src = DATA_DIR / split / tarea
        if not src.exists():
            continue
        for clase in src.iterdir():
            if clase.is_dir():
                dst = CLEAN / split / tarea / clase.name
                n = deduplicar_y_limpiar(clase, dst)
                print(f"{split}/{tarea}/{clase.name}: {n}")


In [ ]:
from sklearn.model_selection import train_test_split

SPLIT = Path("./splits")
SPLIT.mkdir(exist_ok=True)


def hacer_split(tarea: str):
    base = CLEAN / "Train" / tarea
    out = SPLIT / tarea
    (out / "train").mkdir(parents=True, exist_ok=True)
    (out / "val").mkdir(parents=True, exist_ok=True)
    for clase in base.iterdir():
        if not clase.is_dir():
            continue
        archivos = list(clase.iterdir())
        if len(archivos) < 4:
            print(f"  ATENCION pocos archivos en {clase.name}")
            continue
        tr, va = train_test_split(archivos, test_size=0.2, random_state=42, shuffle=True)
        for split_name, lst in [("train", tr), ("val", va)]:
            dst = out / split_name / clase.name
            dst.mkdir(parents=True, exist_ok=True)
            for f in lst:
                shutil.copy2(f, dst / f.name)
        print(f"  {clase.name}: train={len(tr)} val={len(va)}")


print("Modelo 1 (binario):")
hacer_split("clasificacion_binaria")
print("\nModelo 2 (patogeno):")
hacer_split("clasificacion_patogeno")


In [ ]:
TEST_OUT = SPLIT / "test"
TEST_OUT.mkdir(exist_ok=True)
for tarea in ["clasificacion_binaria", "clasificacion_patogeno"]:
    src = CLEAN / "Test" / tarea
    if src.exists():
        dst = TEST_OUT / tarea
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
print("Test listo en", TEST_OUT)


## Verificar

- `splits/clasificacion_binaria/{train,val}/{soya_sana,soya_enferma}`
- `splits/clasificacion_patogeno/{train,val}/{bacterianas,fungicas,plagas_insectos,roya,virales}`
- `splits/test/clasificacion_{binaria,patogeno}/...`

Prosigue con `02_train_model1_binary.ipynb`.
